# Class 14 - Merging & Reshaping
### Module 3 . Week 5 . Saturday

Real data never arrives in a single table. Demographics live in one file,
exam scores in another, attendance in a third. Joining them correctly -
and knowing what you lose or gain with each join type - is one of the most
important data engineering skills in analytics.

Today we merge three student datasets, explore every join type, reshape
wide data to long and back, and diagnose every row that doesn't match.

Three datasets loaded today:
- `students_demographics.csv` - 20 students (id, name, age, dept, city)
- `students_grades.csv`       - 100 rows (20 students x 5 subjects)
- `students_attendance.csv`   - 17 students (3 students have no record)

## Learning Objectives

- Perform inner, left, right, and outer joins with `pd.merge()`
- Diagnose row-count changes after every merge
- Use `indicator=True` to trace unmatched rows
- Stack DataFrames vertically and horizontally with `pd.concat()`
- Reshape wide data to long with `pd.melt()` and back with `.pivot()`

In [13]:
import pandas as pd
import numpy as np

DATA_DIR = "./datasets"
# pd.set_option("display.max_columns", None)

demo   = pd.read_csv(f"{DATA_DIR}/students_demographics.csv")
grades = pd.read_csv(f"{DATA_DIR}/students_grades.csv")
attend = pd.read_csv(f"{DATA_DIR}/students_attendance.csv")

print(f"demographics: {demo.shape}    grades: {grades.shape}    attendance: {attend.shape}")

demographics: (20, 5)    grades: (100, 3)    attendance: (17, 4)


In [14]:
demo.head(3)

,student_id,name,age,dept,city
0,S100,Ahmad Raza,19,SE,Lahore
1,S101,Sara Khan,23,CS,Karachi
2,S102,Bilal Ahmed,21,SE,Islamabad


In [15]:
grades.head(6)

,student_id,subject,score
0,S100,Mathematics,84
1,S100,Python,65
2,S100,Statistics,79
3,S100,Data Science,87
4,S100,Communication,82
5,S101,Mathematics,84


In [16]:
attend.head(3)

,student_id,classes_attended,total_classes,attendance_pct
0,S100,53,60,88.3
1,S101,49,60,81.7
2,S102,59,60,98.3


## pd.merge() - the core joining function

Four join types. The one you pick determines which rows appear in the result.

In [17]:
#Inner join - only students present in BOTH tables
inner = pd.merge(demo, attend, on="student_id", how="inner")
print(f"demo rows: {len(demo)}  |  attend rows: {len(attend)}  |  inner join rows: {len(inner)}")
print(f"We lost {len(demo) - len(inner)} students who have no attendance record")
inner.head()

demo rows: 20  |  attend rows: 17  |  inner join rows: 17
We lost 3 students who have no attendance record


,student_id,name,age,dept,city,classes_attended,total_classes,attendance_pct
0,S100,Ahmad Raza,19,SE,Lahore,53,60,88.3
1,S101,Sara Khan,23,CS,Karachi,49,60,81.7
2,S102,Bilal Ahmed,21,SE,Islamabad,59,60,98.3
3,S103,Zara Malik,20,CS,Lahore,34,60,56.7
4,S104,Hassan Ali,21,SE,Faisalabad,46,60,76.7


In [18]:
# Left join - keep ALL students from demo, NaN where attendance is missing
left = pd.merge(demo, attend, on="student_id", how="left")
print(f"left join rows: {len(left)}  (all {len(demo)} students preserved)")
print(f"NaN attendance rows: {left['attendance_pct'].isna().sum()}")
left.tail(5)    # the last 3 students will show NaN attendance

left join rows: 20  (all 20 students preserved)
NaN attendance rows: 3


,student_id,name,age,dept,city,classes_attended,total_classes,attendance_pct
15,S115,Fatima Javed,19,CS,Multan,53.0,60.0,88.3
16,S116,Omar Farooq,21,SE,Islamabad,51.0,60.0,85.0
17,S117,Rabia Shahid,21,CS,Faisalabad,NaN,NaN,NaN
18,S118,Imran Khalid,20,SE,Lahore,NaN,NaN,NaN
19,S119,Mehwish Aslam,19,CS,Karachi,NaN,NaN,NaN


In [19]:
# Outer join - keep everything from both tables
outer = pd.merge(demo, attend, on="student_id", how="outer")
print(f"outer join rows: {len(outer)}")
outer

outer join rows: 20


,student_id,name,age,dept,city,classes_attended,total_classes,attendance_pct
0,S100,Ahmad Raza,19,SE,Lahore,53.0,60.0,88.3
1,S101,Sara Khan,23,CS,Karachi,49.0,60.0,81.7
2,S102,Bilal Ahmed,21,SE,Islamabad,59.0,60.0,98.3
3,S103,Zara Malik,20,CS,Lahore,34.0,60.0,56.7
4,S104,Hassan Ali,21,SE,Faisalabad,46.0,60.0,76.7
5,S105,Nida Butt,20,CS,Rawalpindi,45.0,60.0,75.0
6,S106,Usman Sheikh,23,SE,Multan,46.0,60.0,76.7
7,S107,Ayesha Nawaz,23,CS,Islamabad,31.0,60.0,51.7
8,S108,Faisal Iqbal,19,SE,Lahore,44.0,60.0,73.3
9,S109,Mariam Syed,22,CS,Karachi,38.0,60.0,63.3


## Diagnosing a merge with indicator=True

This is the single most useful diagnostic when a merge produces unexpected row counts. Every row gets a label: `both`, `left_only`, or `right_only`.

In [20]:
diagnostic = pd.merge(demo, attend, on="student_id", how="outer", indicator=True)
print("Row source breakdown:")
print(diagnostic["_merge"].value_counts())
print()
print("Students with no attendance record:")
missing_att = diagnostic[diagnostic["_merge"] == "left_only"][["student_id","name","dept"]]
print(missing_att)

Row source breakdown:
_merge
both          17
left_only      3
right_only     0
Name: count, dtype: int64

Students with no attendance record:
   student_id           name dept
17       S117   Rabia Shahid   CS
18       S118   Imran Khalid   SE
19       S119  Mehwish Aslam   CS


## Merging more than two tables

Chain merges sequentially. Check row count after each step.

In [21]:
# Step 1: wide grade summary - pivot grades to one row per student
grade_wide = grades.pivot_table(
    index   = "student_id",
    columns = "subject",
    values  = "score",
    aggfunc = "mean"
).round(1).reset_index()
grade_wide.columns.name = None
print(f"grade_wide: {grade_wide.shape}")
grade_wide.head(3)

grade_wide: (20, 6)


,student_id,Communication,Data Science,Mathematics,Python,Statistics
0,S100,82.0,87.0,84.0,65.0,79.0
1,S101,75.0,60.0,84.0,55.0,87.0
2,S102,53.0,60.0,75.0,78.0,76.0


In [22]:
# Step 2: merge demographics + grade_wide
step1 = pd.merge(demo, grade_wide, on="student_id", how="inner")
print(f"After demo + grades merge: {step1.shape}")

# Step 3: merge + attendance
full = pd.merge(step1, attend[["student_id","attendance_pct"]], on="student_id", how="left")
print(f"After adding attendance: {full.shape}  (left join keeps all 20 students)")
full.head()

After demo + grades merge: (20, 10)
After adding attendance: (20, 11)  (left join keeps all 20 students)


,student_id,name,age,dept,city,Communication,Data Science,Mathematics,Python,Statistics,attendance_pct
0,S100,Ahmad Raza,19,SE,Lahore,82.0,87.0,84.0,65.0,79.0,88.3
1,S101,Sara Khan,23,CS,Karachi,75.0,60.0,84.0,55.0,87.0,81.7
2,S102,Bilal Ahmed,21,SE,Islamabad,53.0,60.0,75.0,78.0,76.0,98.3
3,S103,Zara Malik,20,CS,Lahore,52.0,71.0,84.0,74.0,88.0,56.7
4,S104,Hassan Ali,21,SE,Faisalabad,95.0,62.0,85.0,66.0,76.0,76.7


## pd.concat() - stacking DataFrames

`pd.concat()` is for stacking, not joining. Use it when you have multiple DataFrames with the same columns that need to be combined into one.

In [24]:
grades

,student_id,subject,score
0,S100,Mathematics,84
1,S100,Python,65
2,S100,Statistics,79
3,S100,Data Science,87
4,S100,Communication,82
...,...,...,...
95,S119,Mathematics,62
96,S119,Python,68
97,S119,Statistics,72
98,S119,Data Science,71


In [29]:
# Vertical stack - same columns, different rows
# Suppose we get grade data in two batches (first 10 and last 10 students)
batch_a = grades[grades["student_id"].isin([f"S{100+i}" for i in range(10)])]
batch_b = grades[grades["student_id"].isin([f"S{110+i}" for i in range(10)])]

combined = pd.concat([batch_a, batch_b], axis=0).reset_index(drop = True)
print(f"batch_a: {len(batch_a)} rows  |  batch_b: {len(batch_b)} rows  |  combined: {len(combined)} rows")
combined

batch_a: 50 rows  |  batch_b: 50 rows  |  combined: 100 rows


,student_id,subject,score
0,S100,Mathematics,84
1,S100,Python,65
2,S100,Statistics,79
3,S100,Data Science,87
4,S100,Communication,82
...,...,...,...
95,S119,Mathematics,62
96,S119,Python,68
97,S119,Statistics,72
98,S119,Data Science,71


In [14]:
# Horizontal stack - same rows, different columns
# Add grade_wide columns alongside demo (no key needed - they share the index)
left_part  = demo.set_index("student_id")[["name","dept"]]
right_part = grade_wide.set_index("student_id")[["Mathematics","Python"]]
side_by_side = pd.concat([left_part, right_part], axis=1)
side_by_side.head()

,name,dept,Mathematics,Python
student_id,,,,
S100,Ahmad Raza,SE,84.0,65.0
S101,Sara Khan,CS,84.0,55.0
S102,Bilal Ahmed,SE,75.0,78.0
S103,Zara Malik,CS,84.0,74.0
S104,Hassan Ali,SE,85.0,66.0


## Reshaping: wide to long with pd.melt()

The grades table is already in long (tidy) format - one row per observation. Wide format has one column per variable. `pd.melt()` converts wide to long.

In [15]:
# grade_wide is in wide format: each subject is its own column
print("Wide format:")
grade_wide.head(3)

Wide format:


,student_id,Communication,Data Science,Mathematics,Python,Statistics
0,S100,82.0,87.0,84.0,65.0,79.0
1,S101,75.0,60.0,84.0,55.0,87.0
2,S102,53.0,60.0,75.0,78.0,76.0


In [16]:
# pd.melt() converts to long (tidy) format
grade_long = pd.melt(
    grade_wide,
    id_vars    = ["student_id"],   # columns to keep as-is
    var_name   = "subject",        # name for the new 'variable' column
    value_name = "score",          # name for the new 'value' column
)
print(f"Wide shape: {grade_wide.shape}  ->  Long shape: {grade_long.shape}")
grade_long.sort_values(["student_id","subject"]).head(10)

Wide shape: (20, 6)  ->  Long shape: (100, 3)


,student_id,subject,score
0,S100,Communication,82.0
20,S100,Data Science,87.0
40,S100,Mathematics,84.0
60,S100,Python,65.0
80,S100,Statistics,79.0
1,S101,Communication,75.0
21,S101,Data Science,60.0
41,S101,Mathematics,84.0
61,S101,Python,55.0
81,S101,Statistics,87.0


In [17]:
# Long to wide - the reverse: .pivot()
# Takes long format and spreads one column into many
restored = grade_long.pivot(index="student_id", columns="subject", values="score")
restored.columns.name = None
print(f"Restored shape: {restored.shape}")
restored.head(3)

Restored shape: (20, 5)


,Communication,Data Science,Mathematics,Python,Statistics
student_id,,,,,
S100,82.0,87.0,84.0,65.0,79.0
S101,75.0,60.0,84.0,55.0,87.0
S102,53.0,60.0,75.0,78.0,76.0


## Practical

### Practical 1 - Inner join silently drops rows: always check the count

In [18]:
print("Before merge:")
print(f"  demo rows:   {len(demo)} students")
print(f"  attend rows: {len(attend)} students")

result = pd.merge(demo, attend, on="student_id", how="inner")
print(f"\nAfter inner merge: {len(result)} rows")
print(f"Silently dropped: {len(demo) - len(result)} students!")
print()
print("Which students were dropped?")
dropped_ids = set(demo["student_id"]) - set(result["student_id"])
print(demo[demo["student_id"].isin(dropped_ids)][["student_id","name"]])
print()
print("Always compare shape before and after every merge.")
print("Inner joins silently discard rows with no match - no error, no warning.")

Before merge:
  demo rows:   20 students
  attend rows: 17 students

After inner merge: 17 rows
Silently dropped: 3 students!

Which students were dropped?
   student_id           name
17       S117   Rabia Shahid
18       S118   Imran Khalid
19       S119  Mehwish Aslam

Always compare shape before and after every merge.
Inner joins silently discard rows with no match - no error, no warning.


### Practical 2 - Many-to-many explosion: what happens when both tables have duplicates

In [19]:
# Simulate a many-to-many: student takes a subject in multiple semesters
student_subjects = pd.DataFrame({
    "student_id": ["S100","S100","S101"],
    "subject":    ["Python","Python","Python"],  # S100 has Python TWICE
    "semester":   [1, 2, 1],
})
subject_info = pd.DataFrame({
    "subject":   ["Python","Python"],         # Python appears TWICE in right table
    "teacher":   ["Dr. Ali","Dr. Sara"],
})

result = pd.merge(student_subjects, subject_info, on="subject", how="inner")
print(f"Left rows: {len(student_subjects)}  |  Right rows: {len(subject_info)}")
print(f"Result rows: {len(result)}  <- EXPLOSION: 2x2=4 for Python, 1x2=2 for S101's Python")
print()
print(result)
print()
print("Detect before merging:")
left_dups  = student_subjects.duplicated(subset=["student_id","subject"]).sum()
right_dups = subject_info.duplicated(subset=["subject"]).sum()
print(f"Duplicate keys in left:  {left_dups}")
print(f"Duplicate keys in right: {right_dups}")
print("If either > 0 on the join key, expect a many-to-many explosion.")

Left rows: 3  |  Right rows: 2
Result rows: 6  <- EXPLOSION: 2x2=4 for Python, 1x2=2 for S101's Python

  student_id subject  semester   teacher
0       S100  Python         1   Dr. Ali
1       S100  Python         1  Dr. Sara
2       S100  Python         2   Dr. Ali
3       S100  Python         2  Dr. Sara
4       S101  Python         1   Dr. Ali
5       S101  Python         1  Dr. Sara

Detect before merging:
Duplicate keys in left:  1
Duplicate keys in right: 1
If either > 0 on the join key, expect a many-to-many explosion.


### Practical 3 - melt() is the gateway to every Seaborn plot

In [ ]:
# Seaborn and most visualisation libraries expect LONG (tidy) format.
# Wide format breaks them. Here's why.

# Wide: one column per subject - hard to group/colour by subject
print("Wide (grade_wide) - can't easily colour by subject in a plot:")
print(grade_wide.head(3).to_string())

# Long: every observation is a row - trivial to use as hue= or x= parameter
print("\nLong (grade_long) - one row per observation, perfect for plotting:")
print(grade_long.head(6).to_string())
print()
print("In Week 6 (Seaborn), EVERY plot we build will take the long format.")
print("Get comfortable converting to it now.")

### Practical 4 - concat vs merge: choosing the right tool

In [20]:
# concat: same COLUMNS, different ROWS (stacking more records)
# merge:  shared KEY COLUMN linking different information about the same entity

# When to use concat:
jan = pd.DataFrame({"student_id":["S100"],"month":["Jan"],"score":[88]})
feb = pd.DataFrame({"student_id":["S100"],"month":["Feb"],"score":[92]})
stacked = pd.concat([jan, feb], axis=0)
print("concat (same structure, more rows):")
print(stacked)

# When to use merge:
names_df  = pd.DataFrame({"student_id":["S100"],"name":["Ahmad"]})
scores_df = pd.DataFrame({"student_id":["S100"],"score":[88]})
joined = pd.merge(names_df, scores_df, on="student_id")
print("\nmerge (different info, same student):")
print(joined)
print()
print("Decision rule: stacking more records of the same type -> concat")
print("               combining different facts about the same entity -> merge")

concat (same structure, more rows):
  student_id month  score
0       S100   Jan     88
0       S100   Feb     92

merge (different info, same student):
  student_id   name  score
0       S100  Ahmad     88

Decision rule: stacking more records of the same type -> concat
               combining different facts about the same entity -> merge


## Practical Exercise - Multi-source Student Data Integration
**Difficulty: Medium-High**

In [21]:
# Task 1: merge demographics + grades (inner) then + attendance (left)
# Count row loss at each step and report it
step1 = pd.merge(demo, grade_wide, on="student_id", how="inner")
print(f"After demo + grades (inner):   {step1.shape}")

full_data = pd.merge(step1, attend[["student_id","attendance_pct"]], on="student_id", how="left")
print(f"After + attendance (left):     {full_data.shape}")
print(f"Students missing attendance:   {full_data['attendance_pct'].isna().sum()}")

After demo + grades (inner):   (20, 10)
After + attendance (left):     (20, 11)
Students missing attendance:   3


In [22]:
# Task 2: use indicator=True to find students with no attendance record
diag = pd.merge(demo[["student_id","name"]], attend[["student_id"]], on="student_id",
                how="left", indicator=True)
no_attend = diag[diag["_merge"] == "left_only"][["student_id","name"]]
print(f"Students with no attendance record ({len(no_attend)}):")
print(no_attend.to_string(index=False))

Students with no attendance record (3):
student_id          name
      S117  Rabia Shahid
      S118  Imran Khalid
      S119 Mehwish Aslam


In [23]:
# Task 3: melt full_data to long format with id_vars preserved
id_cols      = ["student_id","name","dept","city","attendance_pct"]
subject_cols = ["Communication","Data Science","Mathematics","Python","Statistics"]

full_long = pd.melt(full_data, id_vars=id_cols, value_vars=subject_cols,
                    var_name="subject", value_name="score")
print(f"Wide shape: {full_data.shape}  ->  Long shape: {full_long.shape}")
full_long.sort_values(["student_id","subject"]).head(6)

Wide shape: (20, 11)  ->  Long shape: (100, 7)


,student_id,name,dept,city,attendance_pct,subject,score
0,S100,Ahmad Raza,SE,Lahore,88.3,Communication,82.0
20,S100,Ahmad Raza,SE,Lahore,88.3,Data Science,87.0
40,S100,Ahmad Raza,SE,Lahore,88.3,Mathematics,84.0
60,S100,Ahmad Raza,SE,Lahore,88.3,Python,65.0
80,S100,Ahmad Raza,SE,Lahore,88.3,Statistics,79.0
1,S101,Sara Khan,CS,Karachi,81.7,Communication,75.0


In [24]:
# Task 4: using the long format - average score per department per subject
dept_subject_avg = (full_long.groupby(["dept","subject"])["score"]
                             .mean()
                             .round(1)
                             .unstack())
print("Average score by department and subject:")
dept_subject_avg

Average score by department and subject:


subject,Communication,Data Science,Mathematics,Python,Statistics
dept,,,,,
CS,67.4,73.0,69.2,66.8,76.3
SE,71.6,68.4,81.3,71.4,76.8


In [25]:
# Task 5: set difference to verify no student IDs are missing after left merge
demo_ids  = set(demo["student_id"])
full_ids  = set(full_data["student_id"])
print(f"All demo students in full_data: {demo_ids == full_ids}")
print(f"IDs in demo not in full_data:   {demo_ids - full_ids}")

All demo students in full_data: True
IDs in demo not in full_data:   set()


---
## Summary

| Concept | Key point |
|---|---|
| Inner join | Only matched rows survive - check count before and after |
| Left join | All left rows preserved, NaN where right has no match |
| `indicator=True` | Tags every row as `both`, `left_only`, or `right_only` - essential for diagnostics |
| Many-to-many | Duplicate keys on both sides cause row explosion - detect with `.duplicated()` first |
| `pd.concat(axis=0)` | Stack more rows - same column structure required |
| `pd.melt()` | Wide to long (tidy) - required format for Seaborn and most analytical functions |
| `.pivot()` | Long to wide - the inverse of melt |

## Homework

Predict: what does `pd.merge(A, B, on='id', how='right')` keep when `B` has a row
whose `id` doesn't appear in `A`? Write your answer before verifying.